# YRBSS 2023 Geographic Data Cleaning

This notebook prepares the 2023 combined YRBSS state and district datasets for geographic analysis.

The cleaning workflow will:
- inspect the combined state and district fixed-width files,
- use the CDC SAS input program to identify field positions,
- retain geographic identifiers, survey design fields, selected mental-health outcomes, and selected risk/protective factors,
- preserve state and district survey levels separately,
- validate the cleaned outputs before export.

In [13]:
from pathlib import Path

geo_dir = Path("../../data/raw/YRBSS_GEO")

files = sorted(geo_dir.iterdir())

for file in files:
    print(file.name)
    import pandas as pd
from pathlib import Path

2023-SADC-SAS-Input-Program.sas
2023-YRBS-SADC-Documentation.pdf
sadc_2023_district.dat
sadc_2023_state_a_d.dat
sadc_2023_state_e_h.dat
sadc_2023_state_i_l.dat
sadc_2023_state_m.dat
sadc_2023_state_n_p.dat
sadc_2023_state_q_t.dat
sadc_2023_state_u_z.dat


## Inspect SAS Input Program

The CDC SAS input program is used to identify the fixed-width column positions and variable names needed to decode the combined geographic YRBSS files.

In [14]:
sas_path = geo_dir / "2023-SADC-SAS-Input-Program.sas"

with open(sas_path, "r", encoding="utf-8", errors="ignore") as f:
    sas_text = f.read()

print(sas_text[:12000])

/****************************************************************************************/
/*  This SAS program reads ASCII format (text format) 2023 SADC data and creates a      */
/*  formatted and labeled SAS dataset.                                                  */
/*                                                                                      */
/*  Change the file location specifications from 'c:\sadc2023' to the location where    */
/*  you downloaded, unzipped, and stored the YRBS ASCII data file and the format        */
/*  library before you run this program.  Change the location specification in three    */
/*  places - in the 'filename' statement and in the two 'libname' statements at the     */
/*  top of the program.                                                                 */
/*                                                                                      */
/*  Change 'xxxxxxx' in the 'filename' statement and the 'data' statement to            */

## Validate Fixed-Width Geographic Fields

The SAS input program confirms that the combined YRBSS files contain explicit site identifiers and survey-type fields. Before importing the full files, sample records are decoded to verify that the documented fixed-width positions correctly identify district and state survey records.

In [15]:
district_path = geo_dir / "sadc_2023_district.dat"
state_path = geo_dir / "sadc_2023_state_a_d.dat"

def inspect_record(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        line = f.readline().rstrip("\n")

    print("File:", path.name)
    print("Record length:", len(line))
    print("Site code:", line[0:5].strip())
    print("Site name:", line[5:55].strip())
    print("Site type:", line[55:105].strip())
    print("Site type number:", line[105:113].strip())
    print("Year:", line[113:121].strip())
    print("Weight:", line[124:134].strip())
    print("Stratum:", line[134:142].strip())
    print("PSU:", line[142:150].strip())
    print()

inspect_record(district_path)
inspect_record(state_path)

File: sadc_2023_district.dat
Record length: 775
Site code: CH
Site name: Chicago, IL (CH)
Site type: District
Site type number: 1
Year: 1991
Weight: 76.7361
Stratum: 1
PSU: 14

File: sadc_2023_state_a_d.dat
Record length: 775
Site code: AL
Site name: Alabama (AL)
Site type: State
Site type number: 2
Year: 1991
Weight: 98.1638
Stratum: 8
PSU: 2



### Validation Result

The documented fixed-width positions correctly identify district and state survey records. The combined geographic files contain multiple historical survey years.

For this project, the cleaned geographic dataset will retain the 2019, 2021, and 2023 survey cycles. The 2023 survey will serve as the primary geographic snapshot, while 2019 and 2021 provide recent trend and coverage context.

In [16]:
# Fixed-width positions from the CDC SAS input program
# Python uses zero-based indexing, so SAS positions are converted accordingly.

colspecs = [
    (0, 5),      # sitecode
    (5, 55),     # sitename
    (55, 105),   # sitetype
    (105, 113),  # sitetypenum
    (113, 121),  # year
    (124, 134),  # weight
    (134, 142),  # stratum
    (142, 150),  # psu
    (150, 158),  # record
]

names = [
    "sitecode",
    "sitename",
    "sitetype",
    "sitetypenum",
    "year",
    "weight",
    "stratum",
    "psu",
    "record",
]

district_geo = pd.read_fwf(
    district_path,
    colspecs=colspecs,
    names=names
)

state_geo = pd.read_fwf(
    state_path,
    colspecs=colspecs,
    names=names
)

print("District file shape:", district_geo.shape)
print("State A-D file shape:", state_geo.shape)

print("\nDistrict years:")
print(district_geo["year"].value_counts().sort_index())

print("\nState A-D years:")
print(state_geo["year"].value_counts().sort_index())

District file shape: (522328, 9)
State A-D file shape: (105455, 9)

District years:
year
1991    10404
1993    12650
1995    16207
1997    16278
1999    15990
2001    16621
2003    35692
2005    37440
2007    38586
2009    48810
2011    48818
2013    41529
2015    39400
2017    44095
2019    47305
2021    30375
2023    22128
Name: count, dtype: int64

State A-D years:
year
1991     2477
1993     4412
1995     7812
1997     5778
1999     3549
2001     3272
2003     6000
2005     7560
2007     6021
2009     8690
2011     8482
2013     5979
2015    10388
2017     8424
2019    10547
2021     4188
2023     1876
Name: count, dtype: int64


In [17]:
import pandas as pd

## Restrict Geographic Data to Recent Survey Years

The combined YRBSS geographic files contain records from multiple historical survey cycles. For this analysis, state and district records are restricted to 2019, 2021, and 2023.

The 2023 survey serves as the primary geographic snapshot, while 2019–2023 provides recent trend context.

State and district survey records remain separate geographic levels throughout the analysis.

In [18]:
# Recent YRBSS survey cycles used for geographic analysis

analysis_years = [2019, 2021, 2023]

# All combined state files
state_paths = [
    geo_dir / "sadc_2023_state_a_d.dat",
    geo_dir / "sadc_2023_state_e_h.dat",
    geo_dir / "sadc_2023_state_i_l.dat",
    geo_dir / "sadc_2023_state_m.dat",
    geo_dir / "sadc_2023_state_n_p.dat",
    geo_dir / "sadc_2023_state_q_t.dat",
    geo_dir / "sadc_2023_state_u_z.dat",
]

# Load geography/design fields from every state file
state_frames = []

for path in state_paths:
    temp = pd.read_fwf(
        path,
        colspecs=colspecs,
        names=names
    )

    temp["source_file"] = path.name
    state_frames.append(temp)

all_states_geo = pd.concat(
    state_frames,
    ignore_index=True
)

# Restrict district and state records to selected years
district_recent_geo = district_geo[
    district_geo["year"].isin(analysis_years)
].copy()

states_recent_geo = all_states_geo[
    all_states_geo["year"].isin(analysis_years)
].copy()

print("Analysis years:", analysis_years)

print("\nRecent district records:", len(district_recent_geo))
print("Recent state records:", len(states_recent_geo))

print("\nDistrict records by year:")
print(
    district_recent_geo["year"]
    .value_counts()
    .sort_index()
)

print("\nState records by year:")
print(
    states_recent_geo["year"]
    .value_counts()
    .sort_index()
)

print("\nUnique district jurisdictions by year:")
print(
    district_recent_geo
    .groupby("year")["sitename"]
    .nunique()
)

print("\nUnique state jurisdictions by year:")
print(
    states_recent_geo
    .groupby("year")["sitename"]
    .nunique()
)

print("\nDistrict jurisdictions represented:")
print(
    sorted(
        district_recent_geo["sitename"]
        .dropna()
        .unique()
    )
)

print("\nState jurisdictions represented:")
print(
    sorted(
        states_recent_geo["sitename"]
        .dropna()
        .unique()
    )
)

Analysis years: [2019, 2021, 2023]

Recent district records: 99808
Recent state records: 403042

District records by year:
year
2019    47305
2021    30375
2023    22128
Name: count, dtype: int64

State records by year:
year
2019    159198
2021    130025
2023    113819
Name: count, dtype: int64

Unique district jurisdictions by year:
year
2019    22
2021    17
2023    10
Name: sitename, dtype: int64

Unique state jurisdictions by year:
year
2019    37
2021    35
2023    28
Name: sitename, dtype: int64

District jurisdictions represented:
['Albuquerque, NM (AB)', 'Borough of Bronx, NY (NYG)', 'Borough of Brooklyn, NY (NYH)', 'Borough of Manhattan, NY (NYI)', 'Borough of Queens, NY (NYJ)', 'Borough of Staten Island, NY (NYK)', 'Broward County, FL (FT)', 'Chicago, IL (CH)', 'Duval County, FL (DU)', 'Fort Worth, TX (FW)', 'Hillsborough County, FL (HL)', 'Los Angeles, CA (LO)', 'New York City, NY (NYC)', 'Oakland, CA (OA)', 'Orange County, FL (OL)', 'Palm Beach County, FL (PB)', 'Pasco Coun

## Decode Selected Mental Health and Context Variables

The geographic YRBSS files are now restricted to the 2019, 2021, and 2023 survey cycles.

The next step is to decode a focused set of variables that align with the national YRBSS analysis. These include:

- Mental health outcomes
- Suicide-related outcomes
- Bullying
- Sleep
- Selected adverse childhood experiences
- School and family protective factors

The raw coded responses are retained at this stage so that response meanings and availability can be validated before recoding.

In [19]:
# Final selected fixed-width fields from the CDC SAS input program

selected_colspecs = [
    # Geography / survey design
    (0, 5),       # sitecode
    (5, 55),      # sitename
    (55, 105),    # sitetype
    (105, 113),   # sitetypenum
    (113, 121),   # year
    (124, 134),   # weight
    (134, 142),   # stratum
    (142, 150),   # psu
    (150, 158),   # record

    # Mental health / bullying outcomes
    (278, 279),   # q24 bullied at school
    (279, 280),   # q25 electronic bullying
    (280, 281),   # q26 sad or hopeless
    (281, 282),   # q27 considered suicide
    (282, 283),   # q28 made suicide plan
    (283, 284),   # q29 attempted suicide

    # Mental health / protective behavior
    (335, 336),   # q84 current mental health
    (336, 337),   # q85 sleep

    # ACE / household / school context
    (657, 658),   # qbasicneedsace
    (661, 662),   # qemoabuseace
    (663, 664),   # qincarparentace
    (664, 665),   # qintviolenceace
    (668, 669),   # qparentalmonitoring
    (669, 670),   # qphyabuseace
    (671, 672),   # qsexabuseace
    (679, 680),   # qunfairlydisc
]

selected_names = [
    "sitecode",
    "sitename",
    "sitetype",
    "sitetypenum",
    "year",
    "weight",
    "stratum",
    "psu",
    "record",

    "bullied_at_school",
    "electronically_bullied",
    "persistent_sadness_hopelessness",
    "considered_suicide",
    "made_suicide_plan",
    "attempted_suicide",

    "poor_mental_health",
    "sleep",

    "adult_met_basic_needs",
    "emotional_abuse",
    "parent_guardian_incarceration",
    "witnessed_intimate_partner_violence",
    "parental_monitoring",
    "physical_abuse",
    "sexual_abuse",
    "unfair_discipline_at_school",
]

print("Selected variables:", len(selected_names))
print(selected_names)

Selected variables: 25
['sitecode', 'sitename', 'sitetype', 'sitetypenum', 'year', 'weight', 'stratum', 'psu', 'record', 'bullied_at_school', 'electronically_bullied', 'persistent_sadness_hopelessness', 'considered_suicide', 'made_suicide_plan', 'attempted_suicide', 'poor_mental_health', 'sleep', 'adult_met_basic_needs', 'emotional_abuse', 'parent_guardian_incarceration', 'witnessed_intimate_partner_violence', 'parental_monitoring', 'physical_abuse', 'sexual_abuse', 'unfair_discipline_at_school']


### Verify Selected Variable Positions

Before decoding the full geographic files, the SAS input program is searched for the mental-health, sleep, ACE, and protective-factor variables. This confirms the exact fixed-width positions used by CDC and identifies which supplemental variables can be retained for analysis.

In [20]:
# Find exact SAS input lines for selected mental-health,
# ACE, and protective-factor variables

search_vars = [
    "q24",
    "q25",
    "q26",
    "q27",
    "q28",
    "q29",
    "q84",
    "q85",
    "qemoabuseace",
    "qincarparentace",
    "qintviolenceace",
    "qparentalmonitoring",
    "qphyabuseace",
    "qsexabuseace",
    "qunfairlydisc",
    "qbasicneedsace",
]

sas_lines = sas_text.splitlines()

for var in search_vars:
    print(f"\n--- {var} ---")
    
    matches = [
        line.strip()
        for line in sas_lines
        if var.lower() in line.lower()
    ]
    
    if matches:
        for line in matches:
            print(line)
    else:
        print("Not found")


--- q24 ---
q24 $ 279-279
q24 $H24S.
q24="Bullying at school"

--- q25 ---
q25 $ 280-280
q25 $H25S.
q25="Electronic bullying"

--- q26 ---
q26 $ 281-281
q26 $H26S.
q26="Sad or hopeless"

--- q27 ---
q27 $ 282-282
q27 $H27S.
q27="Considered suicide"

--- q28 ---
q28 $ 283-283
q28 $H28S.
q28="Made a suicide plan"

--- q29 ---
q29 $ 284-284
q29 $H29S.
q29="Attempted suicide"

--- q84 ---
q84 $ 336-336
q84 $H84S.
q84="Current mental health"

--- q85 ---
q85 $ 337-337
q85 $H85S.
q85="Sleep"

--- qemoabuseace ---
qemoabuseace $ 662-662
qemoabuseace $ACE2F.
qemoabuseace="Parental emotional abuse ACEs"

--- qincarparentace ---
qincarparentace $ 664-664
qincarparentace $ACE1F.
qincarparentace="Ever incarcerated parent/guardian ACEs"

--- qintviolenceace ---
qintviolenceace $ 665-665
qintviolenceace $ACE2F.
qintviolenceace="Adults in home intimate partner violence ACEs"

--- qparentalmonitoring ---
qparentalmonitoring $ 669-669
qparentalmonitoring $ACE2F.
qparentalmonitoring="Parental monitorin

## Load Selected Geographic YRBSS Variables

Using the verified CDC fixed-width positions, the selected mental-health, bullying, ACE, protective-factor, geography, and survey-design variables are decoded from the combined district and state files.

Only records from the 2019, 2021, and 2023 survey cycles are retained.

In [21]:
# Load selected variables from the district file

district_selected = pd.read_fwf(
    district_path,
    colspecs=selected_colspecs,
    names=selected_names,
    dtype=str
)

# Load selected variables from all state files

state_selected_frames = []

for path in state_paths:
    temp = pd.read_fwf(
        path,
        colspecs=selected_colspecs,
        names=selected_names,
        dtype=str
    )
    
    temp["source_file"] = path.name
    state_selected_frames.append(temp)

states_selected = pd.concat(
    state_selected_frames,
    ignore_index=True
)

# Convert numeric fields
numeric_cols = [
    "sitetypenum",
    "year",
    "weight",
    "stratum",
    "psu",
    "record",
]

for col in numeric_cols:
    district_selected[col] = pd.to_numeric(
        district_selected[col],
        errors="coerce"
    )
    
    states_selected[col] = pd.to_numeric(
        states_selected[col],
        errors="coerce"
    )

# Restrict to recent analysis years

district_recent = district_selected[
    district_selected["year"].isin(analysis_years)
].copy()

states_recent = states_selected[
    states_selected["year"].isin(analysis_years)
].copy()

print("District recent shape:", district_recent.shape)
print("State recent shape:", states_recent.shape)

print("\nDistrict years:")
print(district_recent["year"].value_counts().sort_index())

print("\nState years:")
print(states_recent["year"].value_counts().sort_index())

District recent shape: (99808, 25)
State recent shape: (403042, 26)

District years:
year
2019    47305
2021    30375
2023    22128
Name: count, dtype: int64

State years:
year
2019    159198
2021    130025
2023    113819
Name: count, dtype: int64


## Check Variable Availability by Survey Year

YRBSS state and district questionnaires can vary across jurisdictions and survey cycles. Before recoding responses, variable availability is assessed by year to determine which measures can support trend analysis and which should be treated as 2023-only or limited-coverage measures.

In [22]:
analysis_vars = [
    "bullied_at_school",
    "electronically_bullied",
    "persistent_sadness_hopelessness",
    "considered_suicide",
    "made_suicide_plan",
    "attempted_suicide",
    "poor_mental_health",
    "sleep",
    "adult_met_basic_needs",
    "emotional_abuse",
    "parent_guardian_incarceration",
    "witnessed_intimate_partner_violence",
    "parental_monitoring",
    "physical_abuse",
    "sexual_abuse",
    "unfair_discipline_at_school",
]

def availability_by_year(df, label):
    print(f"\n{label}")
    print("=" * len(label))

    for year in analysis_years:
        temp = df[df["year"] == year]

        print(f"\nYEAR: {year}")
        print(f"Records: {len(temp):,}")

        for var in analysis_vars:
            nonmissing = temp[var].notna().sum()
            pct = (nonmissing / len(temp) * 100) if len(temp) else 0

            print(
                f"{var:<38} "
                f"{nonmissing:>8,} "
                f"({pct:>5.1f}%)"
            )

availability_by_year(
    district_recent,
    "DISTRICT VARIABLE AVAILABILITY"
)

availability_by_year(
    states_recent,
    "STATE VARIABLE AVAILABILITY"
)


DISTRICT VARIABLE AVAILABILITY

YEAR: 2019
Records: 47,305
bullied_at_school                        46,067 ( 97.4%)
electronically_bullied                   46,250 ( 97.8%)
persistent_sadness_hopelessness          45,871 ( 97.0%)
considered_suicide                       44,981 ( 95.1%)
made_suicide_plan                        26,102 ( 55.2%)
attempted_suicide                        38,601 ( 81.6%)
poor_mental_health                            0 (  0.0%)
sleep                                    40,806 ( 86.3%)
adult_met_basic_needs                         0 (  0.0%)
emotional_abuse                               0 (  0.0%)
parent_guardian_incarceration                 0 (  0.0%)
witnessed_intimate_partner_violence           0 (  0.0%)
parental_monitoring                           0 (  0.0%)
physical_abuse                                0 (  0.0%)
sexual_abuse                                  0 (  0.0%)
unfair_discipline_at_school                   0 (  0.0%)

YEAR: 2021
Records: 30,375


### Variable Coverage Decision

Variable availability differs substantially across survey years and jurisdictions because state and district YRBSS questionnaires do not include every supplemental question consistently.

For geographic analysis:

**2019–2023 trend measures**
- Bullying at school
- Electronic bullying
- Persistent sadness/hopelessness
- Considered suicide
- Suicide plan
- Suicide attempt
- Sleep

**2021–2023 contextual measures**
- Poor mental health
- Adult support for basic needs
- Parent/guardian incarceration
- Witnessed intimate partner violence
- Physical abuse
- Sexual abuse

**2023-focused measure**
- Emotional abuse

Parental monitoring and unfair discipline at school have insufficient geographic coverage for primary analysis and will not be used as core geographic measures.

Missing values are preserved because they may represent questions that were not administered in a particular jurisdiction or survey cycle rather than negative responses.

In [23]:
# Inspect raw response codes for retained analysis variables

retained_vars = [
    "bullied_at_school",
    "electronically_bullied",
    "persistent_sadness_hopelessness",
    "considered_suicide",
    "made_suicide_plan",
    "attempted_suicide",
    "poor_mental_health",
    "sleep",
    "adult_met_basic_needs",
    "emotional_abuse",
    "parent_guardian_incarceration",
    "witnessed_intimate_partner_violence",
    "physical_abuse",
    "sexual_abuse",
]

for var in retained_vars:
    print(f"\n{'=' * 60}")
    print(var.upper())
    print("=" * 60)

    print("\nDistrict codes:")
    print(
        district_recent[var]
        .value_counts(dropna=False)
        .sort_index()
    )

    print("\nState codes:")
    print(
        states_recent[var]
        .value_counts(dropna=False)
        .sort_index()
    )


BULLIED_AT_SCHOOL

District codes:
bullied_at_school
1      14041
2      79815
NaN     5952
Name: count, dtype: int64

State codes:
bullied_at_school
1       63327
2      276838
NaN     62877
Name: count, dtype: int64

ELECTRONICALLY_BULLIED

District codes:
electronically_bullied
1      13285
2      84592
NaN     1931
Name: count, dtype: int64

State codes:
electronically_bullied
1       64639
2      334779
NaN      3624
Name: count, dtype: int64

PERSISTENT_SADNESS_HOPELESSNESS

District codes:
persistent_sadness_hopelessness
1      37656
2      59531
NaN     2621
Name: count, dtype: int64

State codes:
persistent_sadness_hopelessness
1      144904
2      251094
NaN      7044
Name: count, dtype: int64

CONSIDERED_SUICIDE

District codes:
considered_suicide
1      17132
2      76559
NaN     6117
Name: count, dtype: int64

State codes:
considered_suicide
1       68072
2      274228
NaN     60742
Name: count, dtype: int64

MADE_SUICIDE_PLAN

District codes:
made_suicide_plan
1       95

## Recode YRBSS Response Categories

The fixed-width geographic files store survey responses as numeric codes. These codes are converted to readable categorical labels using the CDC YRBSS response structure.

Missing values remain missing because some questions were not administered in every jurisdiction or survey year.

In [24]:
# Response-code mappings for retained YRBSS variables

binary_yes_no = {
    "1": "Yes",
    "2": "No",
}

attempt_map = {
    "1": "0 times",
    "2": "1 time",
    "3": "2 or 3 times",
    "4": "4 or 5 times",
    "5": "6 or more times",
}

poor_mental_health_map = {
    "1": "Never",
    "2": "Rarely",
    "3": "Sometimes",
    "4": "Most of the time",
    "5": "Always",
}

sleep_map = {
    "1": "4 or fewer hours",
    "2": "5 hours",
    "3": "6 hours",
    "4": "7 hours",
    "5": "8 hours",
    "6": "9 hours",
    "7": "10 or more hours",
}

frequency_map = {
    "1": "Never",
    "2": "Rarely",
    "3": "Sometimes",
    "4": "Most of the time",
    "5": "Always",
}

basic_needs_map = {
    "1": "Never",
    "2": "Rarely",
    "3": "Sometimes",
    "4": "Most of the time",
    "5": "Always",
}

# Variables using Yes / No coding
binary_vars = [
    "bullied_at_school",
    "electronically_bullied",
    "persistent_sadness_hopelessness",
    "considered_suicide",
    "made_suicide_plan",
    "parent_guardian_incarceration",
    "sexual_abuse",
]

def recode_geographic_yrbss(df):
    cleaned = df.copy()

    # Binary variables
    for var in binary_vars:
        cleaned[var] = cleaned[var].map(binary_yes_no)

    # Ordered / frequency variables
    cleaned["attempted_suicide"] = (
        cleaned["attempted_suicide"]
        .map(attempt_map)
    )

    cleaned["poor_mental_health"] = (
        cleaned["poor_mental_health"]
        .map(poor_mental_health_map)
    )

    cleaned["sleep"] = (
        cleaned["sleep"]
        .map(sleep_map)
    )

    cleaned["adult_met_basic_needs"] = (
        cleaned["adult_met_basic_needs"]
        .map(basic_needs_map)
    )

    cleaned["emotional_abuse"] = (
        cleaned["emotional_abuse"]
        .map(frequency_map)
    )

    cleaned["witnessed_intimate_partner_violence"] = (
        cleaned["witnessed_intimate_partner_violence"]
        .map(frequency_map)
    )

    cleaned["physical_abuse"] = (
        cleaned["physical_abuse"]
        .map(frequency_map)
    )

    return cleaned


district_clean = recode_geographic_yrbss(
    district_recent
)

states_clean = recode_geographic_yrbss(
    states_recent
)

print("District cleaned shape:", district_clean.shape)
print("State cleaned shape:", states_clean.shape)

District cleaned shape: (99808, 25)
State cleaned shape: (403042, 26)


## Validate Recoded Response Values

Recoded categorical values are checked against the original raw response codes to confirm that the cleaning logic preserved the expected response structure.

In [25]:
# Compare raw and recoded values for selected variables

validation_vars = [
    "persistent_sadness_hopelessness",
    "considered_suicide",
    "attempted_suicide",
    "poor_mental_health",
    "sleep",
    "adult_met_basic_needs",
    "emotional_abuse",
    "physical_abuse",
    "sexual_abuse",
]

for var in validation_vars:
    print(f"\n{'=' * 60}")
    print(var.upper())
    print("=" * 60)

    comparison = pd.DataFrame({
        "raw": district_recent[var],
        "clean": district_clean[var]
    })

    print(
        comparison
        .drop_duplicates()
        .sort_values(
            by=["raw"],
            na_position="last"
        )
        .to_string(index=False)
    )


PERSISTENT_SADNESS_HOPELESSNESS
raw clean
  1   Yes
  2    No
NaN   NaN

CONSIDERED_SUICIDE
raw clean
  1   Yes
  2    No
NaN   NaN

ATTEMPTED_SUICIDE
raw           clean
  1         0 times
  2          1 time
  3    2 or 3 times
  4    4 or 5 times
  5 6 or more times
NaN             NaN

POOR_MENTAL_HEALTH
raw            clean
  1            Never
  2           Rarely
  3        Sometimes
  4 Most of the time
  5           Always
NaN              NaN

SLEEP
raw            clean
  1 4 or fewer hours
  2          5 hours
  3          6 hours
  4          7 hours
  5          8 hours
  6          9 hours
  7 10 or more hours
NaN              NaN

ADULT_MET_BASIC_NEEDS
raw            clean
  1            Never
  2           Rarely
  3        Sometimes
  4 Most of the time
  5           Always
NaN              NaN

EMOTIONAL_ABUSE
raw            clean
  1            Never
  2           Rarely
  3        Sometimes
  4 Most of the time
  5           Always
NaN              NaN

PHYSICAL_A

## Final Validation and Export

The cleaned state and district datasets are validated for survey year, geography type, duplicate records, and expected dimensions before export.

State and district records are exported separately so that their survey designs and geographic levels remain distinct during analysis.

In [26]:
# Final validation checks

print("DISTRICT DATA")
print("=" * 50)

print("Shape:", district_clean.shape)
print("Years:", sorted(district_clean["year"].dropna().unique()))
print("Site types:", district_clean["sitetype"].value_counts(dropna=False).to_dict())
print("Unique jurisdictions:", district_clean["sitename"].nunique())

district_duplicates = district_clean.duplicated(
    subset=["sitecode", "year", "record"]
).sum()

print("Duplicate respondent records:", district_duplicates)


print("\nSTATE DATA")
print("=" * 50)

print("Shape:", states_clean.shape)
print("Years:", sorted(states_clean["year"].dropna().unique()))
print("Site types:", states_clean["sitetype"].value_counts(dropna=False).to_dict())
print("Unique jurisdictions:", states_clean["sitename"].nunique())

state_duplicates = states_clean.duplicated(
    subset=["sitecode", "year", "record"]
).sum()

print("Duplicate respondent records:", state_duplicates)


print("\nMISSING SURVEY WEIGHTS")
print("=" * 50)

print(
    "District missing weights:",
    district_clean["weight"].isna().sum()
)

print(
    "State missing weights:",
    states_clean["weight"].isna().sum()
)

DISTRICT DATA
Shape: (99808, 25)
Years: [np.int64(2019), np.int64(2021), np.int64(2023)]
Site types: {'District': 99808}
Unique jurisdictions: 22
Duplicate respondent records: 0

STATE DATA
Shape: (403042, 26)
Years: [np.int64(2019), np.int64(2021), np.int64(2023)]
Site types: {'State': 403042}
Unique jurisdictions: 38
Duplicate respondent records: 0

MISSING SURVEY WEIGHTS
District missing weights: 0
State missing weights: 0


## Cleaning Summary

The 2023 YRBSS state and district combined files were cleaned and prepared for geographic analysis.

### What was completed

- Verified the CDC fixed-width structure using the official SAS input program.
- Identified and validated geographic fields for:
  - District
  - State
- Confirmed that the combined files contain multiple historical survey cycles.
- Restricted the analysis dataset to the recent YRBSS cycles:
  - 2019
  - 2021
  - 2023
- Preserved state and district records as separate geographic levels.
- Retained survey-design fields including:
  - survey weight
  - stratum
  - PSU
- Selected mental-health, suicide, bullying, sleep, ACE, and protective/context variables aligned with the national YRBSS analysis.
- Evaluated variable availability by survey year and geographic level.
- Preserved missing values because some supplemental questions were not administered consistently across jurisdictions or years.
- Removed low-coverage variables from the core geographic analysis plan where appropriate.
- Converted CDC numeric response codes into readable categorical labels.
- Validated raw-to-clean response mappings.
- Checked for duplicate respondent records.
- Confirmed that all retained records have survey weights.

### Final cleaned datasets

**District dataset**
- 99,808 respondent records
- 25 variables
- 22 unique district jurisdictions
- Survey years: 2019, 2021, 2023
- Duplicate respondent records: 0
- Missing survey weights: 0

**State dataset**
- 403,042 respondent records
- 26 variables
- 38 unique state jurisdictions
- Survey years: 2019, 2021, 2023
- Duplicate respondent records: 0
- Missing survey weights: 0

The cleaned datasets are ready for geographic YRBSS analysis. State and district results should continue to be analyzed separately because they represent distinct survey samples and geographic levels.

In [27]:
# Export cleaned YRBSS geographic datasets

cleaned_dir = Path("../../data/cleaned")
cleaned_dir.mkdir(parents=True, exist_ok=True)

district_output = (
    cleaned_dir / "yrbss_2019_2023_district_cleaned.csv"
)

state_output = (
    cleaned_dir / "yrbss_2019_2023_state_cleaned.csv"
)

district_clean.to_csv(
    district_output,
    index=False
)

states_clean.to_csv(
    state_output,
    index=False
)

print("Export complete.")
print("District:", district_output)
print("State:", state_output)

print("\nDistrict shape:", district_clean.shape)
print("State shape:", states_clean.shape)

Export complete.
District: ..\..\data\cleaned\yrbss_2019_2023_district_cleaned.csv
State: ..\..\data\cleaned\yrbss_2019_2023_state_cleaned.csv

District shape: (99808, 25)
State shape: (403042, 26)
